# --- Preprocessing ---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Dropout

col_names = [
    'Team_ID', 'Jaar', 'Maandkey', 'Totaal_FTE_Ziek', 'Totaal_FTE_Toegewezen',
    'Verzuimpercentage', 'Teamomvang', 'Gem_Leeftijd', 'Aandeel_Vrouw',
    'Medewerkers_Met_Overwerk', 'Gem_Contracturen_Week', 'Aandeel_Vast_Contract',
    'Totaal_Overwerk_Uren', 'Gem_Overwerk_Per_Medewerker'
]

df = pd.read_csv("Dataset clean.csv", header=None, names=col_names, decimal=',')

df = df.sort_values(['Team_ID', 'Maandkey']).reset_index(drop=True)

df['Lag_1']  = df.groupby('Team_ID')['Verzuimpercentage'].shift(1)


df = df.dropna(subset=['Lag_1'])

features = [
    'Lag_1', 'Teamomvang', 'Gem_Leeftijd', 'Aandeel_Vrouw',
    'Gem_Contracturen_Week', 'Aandeel_Vast_Contract',
    'Gem_Overwerk_Per_Medewerker'
]


X = df[features]
y = df['Verzuimpercentage']

train = df[df['Maandkey'] <= 202412]
val   = df[(df['Maandkey'] >= 202501) & (df['Maandkey'] <= 202512)]
test  = df[df['Maandkey'] >= 202601]

X_train, y_train = train[features], train['Verzuimpercentage']
X_val,   y_val   = val[features],   val['Verzuimpercentage']
X_test,  y_test  = test[features],  test['Verzuimpercentage']

# --- Model Training ---

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model = Sequential([
    Dense(128, activation='relu', input_shape=(9,)),
    Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history = model.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[es]
)

from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

y_pred_val_ann = model.predict(X_val_scaled).flatten()

mae  = mean_absolute_error(y_val, y_pred_val_ann)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val_ann))
r2   = r2_score(y_val, y_pred_val_ann)

print(f"ANN Validation MAE:  {mae:.4f}")
print(f"ANN Validation RMSE: {rmse:.4f}")
print(f"ANN Validation R²:   {r2:.4f}")

# --- Train Set Evaluation ---

In [ ]:
y_pred_train_ann = model.predict(X_train_scaled).flatten()

mae_train  = mean_absolute_error(y_train, y_pred_train_ann)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train_ann))
r2_train   = r2_score(y_train, y_pred_train_ann)

print(f"ANN Train MAE:  {mae_train:.4f}")
print(f"ANN Train RMSE: {rmse_train:.4f}")
print(f"ANN Train R²:   {r2_train:.4f}")

# --- Try new ANN configuration ---

In [ ]:
model2 = Sequential([
    Dense(128, activation='relu', input_shape=(9,)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

model2.compile(optimizer='adam', loss='mse', metrics=['mae'])

es2 = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history2 = model2.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[es2]
)

print(f"Stopped at epoch: {len(history2.history['loss'])}")

y_pred_val_ann2 = model2.predict(X_val_scaled).flatten()
y_pred_train_ann2 = model2.predict(X_train_scaled).flatten()

print("ANN2 Train:")
print(f"  MAE:  {mean_absolute_error(y_train, y_pred_train_ann2):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train_ann2)):.4f}")
print(f"  R²:   {r2_score(y_train, y_pred_train_ann2):.4f}")

print("\nANN2 Validation:")
print(f"  MAE:  {mean_absolute_error(y_val, y_pred_val_ann2):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_val, y_pred_val_ann2)):.4f}")
print(f"  R²:   {r2_score(y_val, y_pred_val_ann2):.4f}")

y_pred_test_ann2 = model2.predict(X_test_scaled).flatten()

print("ANN2 Test:")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_test_ann2):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test_ann2)):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred_test_ann2):.4f}")

# --- Plot Residual Error ---

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_test_ann2, alpha=0.5, color='steelblue', label='ANN v2')
ax.plot([0, 0.5], [0, 0.5], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Absence Rate')
ax.set_ylabel('Predicted Absence Rate')
ax.set_title('Predicted vs Actual Absence Rate - ANN (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()